In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

train = train.drop(train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)].index)

y_train = np.log1p(train['SalePrice'])

train_features = train.drop(['SalePrice', 'Id'], axis=1)
test_features = test.drop(['Id'], axis=1)
test_ids = test['Id']

ntrain = train_features.shape[0]
all_data = pd.concat([train_features, test_features], axis=0).reset_index(drop=True)

print("Combined shape:", all_data.shape)

Combined shape: (2917, 79)


In [3]:
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
             'MasVnrType']

for col in none_cols:
    all_data[col] = all_data[col].fillna('None')

print(all_data[none_cols].isnull().sum().sum(), "missing values remaining in these columns")

0 missing values remaining in these columns


In [4]:
zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
             'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

print(all_data[zero_cols].isnull().sum().sum(), "missing values remaining in these columns")

0 missing values remaining in these columns


In [5]:
remaining_missing = all_data.isnull().sum()
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)
remaining_missing

LotFrontage    486
MSZoning         4
Utilities        2
Functional       2
Exterior1st      1
Exterior2nd      1
Electrical       1
KitchenQual      1
SaleType         1
dtype: int64

In [6]:
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

print(all_data['LotFrontage'].isnull().sum(), "missing values remaining in LotFrontage")

0 missing values remaining in LotFrontage


In [7]:
mode_cols = ['MSZoning', 'Utilities', 'Functional', 'Exterior1st',
             'Exterior2nd', 'Electrical', 'KitchenQual', 'SaleType']

for col in mode_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

print(all_data[mode_cols].isnull().sum().sum(), "missing values remaining in these columns")

0 missing values remaining in these columns


In [8]:
print("Total missing values left:", all_data.isnull().sum().sum())

Total missing values left: 0


In [9]:
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']

all_data['TotalBath'] = (all_data['FullBath'] + 0.5 * all_data['HalfBath'] +
                          all_data['BsmtFullBath'] + 0.5 * all_data['BsmtHalfBath'])

all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['YearsSinceRemodel'] = all_data['YrSold'] - all_data['YearRemodAdd']

all_data['WasRemodeled'] = (all_data['YearBuilt'] != all_data['YearRemodAdd']).astype(int)

all_data['HasPool'] = (all_data['PoolArea'] > 0).astype(int)
all_data['HasGarage'] = (all_data['GarageArea'] > 0).astype(int)
all_data['HasFireplace'] = (all_data['Fireplaces'] > 0).astype(int)
all_data['Has2ndFloor'] = (all_data['2ndFlrSF'] > 0).astype(int)
all_data['HasBasement'] = (all_data['TotalBsmtSF'] > 0).astype(int)

print("New shape after feature creation:", all_data.shape)

New shape after feature creation: (2917, 89)


In [10]:
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}

qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

for col in qual_cols:
    all_data[col] = all_data[col].map(qual_map)

print(all_data[qual_cols].isnull().sum().sum(), "missing values after mapping")

0 missing values after mapping


In [11]:
exposure_map = {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}
all_data['BsmtExposure'] = all_data['BsmtExposure'].map(exposure_map)

fintype_map = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0}
all_data['BsmtFinType1'] = all_data['BsmtFinType1'].map(fintype_map)
all_data['BsmtFinType2'] = all_data['BsmtFinType2'].map(fintype_map)

garagefin_map = {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0}
all_data['GarageFinish'] = all_data['GarageFinish'].map(garagefin_map)

functional_map = {'Typ': 7, 'Min1': 6, 'Min2': 5, 'Mod': 4, 'Maj1': 3, 'Maj2': 2, 'Sev': 1, 'Sal': 0}
all_data['Functional'] = all_data['Functional'].map(functional_map)

slope_map = {'Gtl': 2, 'Mod': 1, 'Sev': 0}
all_data['LandSlope'] = all_data['LandSlope'].map(slope_map)

paved_map = {'Y': 2, 'P': 1, 'N': 0}
all_data['PavedDrive'] = all_data['PavedDrive'].map(paved_map)

ordinal_cols = ['BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageFinish',
                'Functional', 'LandSlope', 'PavedDrive']

print(all_data[ordinal_cols].isnull().sum().sum(), "missing values after mapping")

0 missing values after mapping


In [12]:
all_data['MSSubClass'] = all_data['MSSubClass'].astype(str)

categorical_cols = all_data.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns to encode:", len(categorical_cols))
print(categorical_cols)

all_data = pd.get_dummies(all_data, columns=categorical_cols, drop_first=True)

print("New shape after one-hot encoding:", all_data.shape)

Categorical columns to encode: 27
['MSSubClass', 'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 'GarageType', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']
New shape after one-hot encoding: (2917, 230)


C:\Users\satya\AppData\Local\Temp\ipykernel_4192\3154273722.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = all_data.select_dtypes(include=['object']).columns.tolist()


In [13]:
from scipy.stats import skew

numeric_feats = all_data.dtypes[all_data.dtypes != "uint8"].index
numeric_feats = all_data[numeric_feats].select_dtypes(include=[np.number]).columns

skewed_feats = all_data[numeric_feats].apply(lambda x: skew(x.dropna()))
skewed_feats = skewed_feats[abs(skewed_feats) > 0.75].sort_values(ascending=False)

print("Number of skewed features:", len(skewed_feats))
skewed_feats

Number of skewed features: 36


MiscVal          21.939672
PoolQC           19.548879
PoolArea         17.688664
HasPool          15.494756
LotArea          13.109495
LowQualFinSF     12.084539
3SsnPorch        11.372080
KitchenAbvGr      4.300550
BsmtFinSF2        4.144503
EnclosedPorch     4.002344
ScreenPorch       3.945101
BsmtHalfBath      3.929996
BsmtFinType2      3.150951
MasVnrArea        2.621719
OpenPorchSF       2.529358
WoodDeckSF        1.844792
ExterCond         1.315069
1stFlrSF          1.257286
BsmtExposure      1.119066
LotFrontage       1.103039
GrLivArea         1.068750
TotalSF           1.009157
BsmtFinSF1        0.980645
BsmtUnfSF         0.919688
2ndFlrSF          0.861556
ExterQual         0.783456
BsmtQual         -1.271611
PavedDrive       -2.977741
GarageQual       -3.262260
GarageCond       -3.381673
BsmtCond         -3.602661
GarageYrBlt      -3.904632
HasGarage        -3.939453
Functional       -4.961675
LandSlope        -4.973254
HasBasement      -5.826825
dtype: float64

In [14]:
all_data[skewed_feats.index] = np.log1p(all_data[skewed_feats.index])

print("Transformation applied to", len(skewed_feats), "columns")
print("Any missing values introduced:", all_data[skewed_feats.index].isnull().sum().sum())

Transformation applied to 36 columns
Any missing values introduced: 0


In [15]:
import joblib
joblib.dump(skewed_feats.index.tolist(), '../models/skewed_cols.pkl')
print("Correct skewed_cols saved:", len(skewed_feats), "columns")

Correct skewed_cols saved: 36 columns


In [16]:
train_processed = all_data[:ntrain]
test_processed = all_data[ntrain:]

print("Train processed shape:", train_processed.shape)
print("Test processed shape:", test_processed.shape)

Train processed shape: (1458, 230)
Test processed shape: (1459, 230)


In [17]:
train_processed.to_csv('../data/train_processed.csv', index=False)
test_processed.to_csv('../data/test_processed.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
test_ids.to_csv('../data/test_ids.csv', index=False)

print("Saved processed data to /data folder")

Saved processed data to /data folder
